# 🦆 DuckDB Getting Started - Simple Demo

A beginner-friendly introduction to DuckDB with step-by-step examples.

## What You'll Learn

This notebook will walk you through:
1. **Basic DuckDB setup and connection**
2. **Loading data from files**
3. **Running SQL queries** 
4. **Comparing with Pandas**
5. **Exporting results**
6. **Advanced features and integration patterns**

Let's get started with your DuckDB journey!

In [ ]:
print("🦆 Welcome to DuckDB!")
print("======================")
print()
print("This notebook will walk you through:")
print("1. Basic DuckDB setup and connection")
print("2. Loading data from files")
print("3. Running SQL queries") 
print("4. Comparing with Pandas")
print("5. Exporting results")

In [ ]:
# Import Libraries
import duckdb
import pandas as pd
import numpy as np
import time
import os
from pathlib import Path

print("📦 Libraries imported successfully!")

# Check data availability
data_dir = Path("/home/jovyan/data")
if data_dir.exists():
    files = list(data_dir.glob("*.parquet"))
    print(f"📁 Found {len(files)} data files:")
    for file in files:
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"  • {file.name}: {size_mb:.1f} MB")
else:
    print("⚠️  Data directory not found. Run data generation first.")

In [ ]:
# Create DuckDB Connection
print("🔌 Connecting to DuckDB...")

# Create an in-memory database
conn = duckdb.connect()

# Configure DuckDB for optimal performance
conn.execute("SET memory_limit='4GB'")
conn.execute("SET threads=4")

print("✅ DuckDB connection established!")
print(f"Memory limit: {conn.execute('SELECT current_setting(\'memory_limit\')').fetchone()[0]}")
print(f"Thread count: {conn.execute('SELECT current_setting(\'threads\')').fetchone()[0]}")

In [ ]:
# Your First Query
print("🎯 Running your first DuckDB query...")

# Simple query to test the connection
result = conn.execute("SELECT 'Hello from DuckDB!' as greeting, 42 as answer").fetchone()
print(f"Greeting: {result[0]}")
print(f"Answer: {result[1]}")

# Show DuckDB version
version = conn.execute("SELECT version()").fetchone()[0]
print(f"DuckDB Version: {version}")

In [ ]:
# Load Data from Files
print("📂 Loading data into DuckDB...")

# Method 1: Create views that reference files directly (recommended)
if data_dir.exists() and (data_dir / "transactions.parquet").exists():
    
    # Create views for each dataset
    conn.execute(f"""
        CREATE VIEW transactions AS 
        SELECT * FROM read_parquet('{data_dir}/transactions.parquet')
    """)
    
    conn.execute(f"""
        CREATE VIEW customers AS 
        SELECT * FROM read_parquet('{data_dir}/customers.parquet')
    """)
    
    # Check what we loaded
    transaction_count = conn.execute("SELECT COUNT(*) FROM transactions").fetchone()[0]
    customer_count = conn.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
    
    print(f"📊 Loaded {transaction_count:,} transactions")
    print(f"👥 Loaded {customer_count:,} customers")
    
    # Show sample data
    print("\n🔍 Sample transactions:")
    sample = conn.execute("SELECT * FROM transactions LIMIT 5").df()
    print(sample)

else:
    print("⚠️  Sample data not found. Creating demo data...")
    
    # Create some demo data
    conn.execute("""
        CREATE TABLE transactions AS
        SELECT 
            row_number() OVER () as transaction_id,
            (random() * 1000)::INTEGER as customer_id,
            CURRENT_DATE - INTERVAL (random() * 365) DAY as transaction_date,
            ['Electronics', 'Clothing', 'Books', 'Food'][
                (random() * 4)::INTEGER + 1
            ] as product_category,
            (random() * 500 + 10)::DECIMAL(10,2) as total_amount
        FROM range(10000)
    """)
    
    print("✅ Created 10,000 demo transactions")

In [ ]:
else:
    print("⚠️  Sample data not found. Creating demo data...")
    
    # Create some demo data
    conn.execute("""
        CREATE TABLE transactions AS
        SELECT 
            row_number() OVER () as transaction_id,
            (random() * 1000)::INTEGER as customer_id,
            CURRENT_DATE - INTERVAL (random() * 365) DAY as transaction_date,
            ['Electronics', 'Clothing', 'Books', 'Food'][
                (random() * 4)::INTEGER + 1
            ] as product_category,
            (random() * 500 + 10)::DECIMAL(10,2) as total_amount
        FROM range(10000)
    """)
    
    print("✅ Created 10,000 demo transactions")

## 🔍 Basic SQL Queries

Now let's run some analytical queries to explore our data. DuckDB supports full SQL syntax with many advanced features.

In [ ]:
# Basic SQL Queries
print("🔍 Running basic analysis queries...")

# Query 1: Total sales by category
print("1️⃣  Sales by Product Category:")
category_sales = conn.execute("""
    SELECT 
        product_category,
        COUNT(*) as transaction_count,
        ROUND(SUM(total_amount), 2) as total_sales,
        ROUND(AVG(total_amount), 2) as avg_transaction_value
    FROM transactions
    GROUP BY product_category
    ORDER BY total_sales DESC
""").df()

print(category_sales)

# Query 2: Top spending customers (if we have customer data)
print("\n2️⃣  Customer Analysis:")
if conn.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_name = 'customers'").fetchone()[0] > 0:
    
    top_customers = conn.execute("""
        SELECT 
            c.customer_id,
            c.customer_segment,
            COUNT(t.transaction_id) as total_transactions,
            ROUND(SUM(t.total_amount), 2) as total_spent,
            ROUND(AVG(t.total_amount), 2) as avg_per_transaction
        FROM customers c
        JOIN transactions t ON c.customer_id = t.customer_id
        GROUP BY c.customer_id, c.customer_segment
        ORDER BY total_spent DESC
        LIMIT 10
    """).df()
    
    print(top_customers)
else:
    
    top_customers = conn.execute("""
        SELECT 
            customer_id,
            COUNT(*) as total_transactions,
            ROUND(SUM(total_amount), 2) as total_spent,
            ROUND(AVG(total_amount), 2) as avg_per_transaction
        FROM transactions
        GROUP BY customer_id
        ORDER BY total_spent DESC
        LIMIT 10
    """).df()
    
    print(top_customers)

## ⚡ Performance Comparison: DuckDB vs Pandas

Let's see how DuckDB performs compared to Pandas for the same analytical query.

In [ ]:
# Performance Comparison: DuckDB vs Pandas
print("⚡ Performance Comparison: DuckDB vs Pandas")
print("=" * 50)

# For this comparison, let's use a reasonable subset
print("Loading data for comparison...")

# DuckDB query timing
start_time = time.time()
duckdb_result = conn.execute("""
    SELECT 
        product_category,
        COUNT(*) as count,
        SUM(total_amount) as total_sales,
        AVG(total_amount) as avg_sales
    FROM transactions
    GROUP BY product_category
    ORDER BY total_sales DESC
""").df()
duckdb_time = time.time() - start_time

print(f"🦆 DuckDB execution time: {duckdb_time:.4f} seconds")

# Pandas equivalent (load data first)
start_time = time.time()

if (data_dir / "transactions.parquet").exists():
    # Load with Pandas
    df_transactions = pd.read_parquet(data_dir / "transactions.parquet")
    
    # Perform the same analysis
    pandas_result = (df_transactions
                    .groupby('product_category')
                    .agg({
                        'transaction_id': 'count',
                        'total_amount': ['sum', 'mean']
                    })
                    .round(2))
    
    pandas_result.columns = ['count', 'total_sales', 'avg_sales']
    pandas_result = pandas_result.reset_index().sort_values('total_sales', ascending=False)
    
else:
    # Use the demo data we created
    df_transactions = conn.execute("SELECT * FROM transactions").df()
    pandas_result = (df_transactions
                    .groupby('product_category')
                    .agg({
                        'transaction_id': 'count', 
                        'total_amount': ['sum', 'mean']
                    })
                    .round(2))

pandas_time = time.time() - start_time

print(f"🐼 Pandas execution time: {pandas_time:.4f} seconds")

if pandas_time > 0:
    speedup = pandas_time / duckdb_time
    print(f"🚀 DuckDB is {speedup:.1f}x faster!")

## 🎯 Advanced DuckDB Features

DuckDB supports many advanced SQL features including window functions, string operations, and complex filtering.

In [ ]:
# Advanced DuckDB Features
print("🎯 Advanced DuckDB Features")
print("=" * 40)

# Window functions
print("1️⃣  Window Functions - Running Totals:")
running_totals = conn.execute("""
    SELECT 
        transaction_date,
        total_amount,
        SUM(total_amount) OVER (
            ORDER BY transaction_date 
            ROWS UNBOUNDED PRECEDING
        ) as running_total,
        ROW_NUMBER() OVER (ORDER BY transaction_date) as transaction_number
    FROM transactions
    ORDER BY transaction_date
    LIMIT 10
""").df()

print(running_totals)

# String operations and filtering
print("\n2️⃣  String Operations and Complex Filtering:")
string_analysis = conn.execute("""
    SELECT 
        product_category,
        LENGTH(product_category) as category_length,
        UPPER(product_category) as category_upper,
        CASE 
            WHEN total_amount > 100 THEN 'High Value'
            WHEN total_amount > 50 THEN 'Medium Value'
            ELSE 'Low Value'
        END as value_tier,
        COUNT(*) as count
    FROM transactions
    WHERE product_category LIKE '%e%'  -- Contains letter 'e'
    GROUP BY product_category, value_tier
    ORDER BY product_category, value_tier
""").df()

print(string_analysis)

## 💾 Data Export Examples

DuckDB makes it easy to export your results to various formats including CSV, Parquet, and JSON.

In [ ]:
# Data Export Examples
print("💾 Exporting Results")
print("=" * 25)

# Export to different formats
output_dir = Path("./output")
output_dir.mkdir(exist_ok=True)

# 1. Export to CSV
category_sales.to_csv(output_dir / "category_sales.csv", index=False)
print("✅ Exported to CSV: category_sales.csv")

# 2. Export to Parquet 
category_sales.to_parquet(output_dir / "category_sales.parquet", index=False)
print("✅ Exported to Parquet: category_sales.parquet")

# 3. Export directly from DuckDB
conn.execute(f"""
    COPY (
        SELECT * FROM transactions 
        WHERE total_amount > 100
        ORDER BY transaction_date DESC
        LIMIT 1000
    ) TO '{output_dir}/high_value_transactions.csv' 
    WITH (HEADER 1, DELIMITER ',')
""")
print("✅ Exported high-value transactions directly from DuckDB")

# 4. Export to JSON
json_result = conn.execute("""
    SELECT 
        product_category,
        JSON_OBJECT(
            'total_sales', SUM(total_amount),
            'transaction_count', COUNT(*),
            'avg_transaction', AVG(total_amount)
        ) as category_stats
    FROM transactions
    GROUP BY product_category
""").df()

json_result.to_json(output_dir / "category_stats.json", orient='records', indent=2)
print("✅ Exported JSON statistics: category_stats.json")

## 🔄 DuckDB ↔ Pandas Integration

One of DuckDB's great features is seamless integration with Pandas DataFrames.

In [ ]:
# Integration with Pandas
print("🔄 DuckDB ↔ Pandas Integration")
print("=" * 40)

# 1. Query DuckDB, get Pandas DataFrame
df_result = conn.execute("""
    SELECT 
        product_category,
        COUNT(*) as transactions,
        SUM(total_amount) as revenue
    FROM transactions
    GROUP BY product_category
""").df()

print("📊 DuckDB query result as Pandas DataFrame:")
print(df_result.dtypes)
print(df_result.head())

# 2. Use Pandas DataFrame in DuckDB query
print("\n🔄 Using Pandas DataFrame in DuckDB:")

# Create a Pandas DataFrame
pandas_data = pd.DataFrame({
    'category': ['Electronics', 'Clothing', 'Books'],
    'target_revenue': [50000, 30000, 15000]
})

print("Pandas DataFrame:")
print(pandas_data)

# Use it directly in DuckDB query
comparison = conn.execute("""
    SELECT 
        t.product_category,
        t.actual_revenue,
        p.target_revenue,
        t.actual_revenue - p.target_revenue as variance,
        ROUND((t.actual_revenue / p.target_revenue) * 100, 1) as achievement_pct
    FROM (
        SELECT 
            product_category,
            SUM(total_amount) as actual_revenue
        FROM transactions
        GROUP BY product_category
    ) t
    JOIN pandas_data p ON t.product_category = p.category
""").df()

print("\nRevenue vs Target Comparison:")
print(comparison)

## 🎉 Summary and Next Steps

Congratulations! You've completed the DuckDB getting started tutorial.

In [ ]:
# Summary and Next Steps
print("🎉 Demo Complete!")
print("=" * 25)

print("What you've learned:")
print("✅ How to connect to DuckDB")
print("✅ Loading data from files") 
print("✅ Running SQL queries")
print("✅ Performance advantages over Pandas")
print("✅ Advanced features (window functions, etc.)")
print("✅ Exporting results to various formats")
print("✅ Pandas integration patterns")

print(f"\n📁 Output files saved to: {output_dir}")
for file in output_dir.glob("*"):
    if file.is_file():
        print(f"  • {file.name}")

print("\n🚀 Next Steps:")
print("1. Explore the advanced performance demo notebook")
print("2. Try your own datasets with DuckDB")  
print("3. Experiment with complex analytical queries")
print("4. Check out DuckDB extensions for geospatial, time series, etc.")

# Clean up
conn.close()
print("\n🔌 DuckDB connection closed.")
print("Happy analyzing! 🦆")